# Simulasi Analisis Data Retail Modern

Notebook ini dibuat untuk membantu pengguna **awam sampai menengah** memahami cara memakai file Excel simulasi retail modern.

**File yang digunakan:** `simulasi_retail_modern_like_indomaret.xlsx`

Notebook ini berisi:
1. Membaca sheet Excel
2. Memahami struktur data
3. Analisis dasar penjualan
4. Analisis per toko dan per kategori
5. Merge / join data
6. Ringkasan KPI level menengah

> Data ini adalah **simulasi**, bukan data asli perusahaan mana pun.

## 1. Persiapan
Pastikan file Excel berada di folder yang sama dengan notebook ini.

Jika belum ada library, install dulu:
```python
!pip install pandas openpyxl matplotlib
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path = 'simulasi_retail_modern_like_indomaret.xlsx'

xls = pd.ExcelFile(file_path)
xls.sheet_names

## 2. Membaca data dari beberapa sheet
Untuk pemula, fokus dulu ke sheet utama berikut:
- `Master_Produk`
- `Master_Toko`
- `Transaksi_Header`
- `Transaksi_Penjualan`
- `Persediaan`

In [ ]:
produk = pd.read_excel(file_path, sheet_name='Master_Produk')
toko = pd.read_excel(file_path, sheet_name='Master_Toko')
trx_header = pd.read_excel(file_path, sheet_name='Transaksi_Header')
trx_detail = pd.read_excel(file_path, sheet_name='Transaksi_Penjualan')
persediaan = pd.read_excel(file_path, sheet_name='Persediaan')

print('Produk:', produk.shape)
print('Toko:', toko.shape)
print('Header Transaksi:', trx_header.shape)
print('Detail Transaksi:', trx_detail.shape)
print('Persediaan:', persediaan.shape)

## 3. Melihat isi data (level awam)
Langkah paling aman saat baru mulai adalah melihat 5 baris pertama.

In [ ]:
produk.head()

In [ ]:
trx_detail.head()

## 4. Memahami kolom penting
Di data retail, beberapa kolom utama yang sering dipakai:
- `SKU` = kode produk
- `Kode Toko` = kode cabang / toko
- `Qty` = jumlah barang terjual
- `Net Sales` = nilai penjualan bersih
- `Kategori` = kelompok produk
- `HPP` = harga pokok penjualan

In [ ]:
trx_detail.columns.tolist()

## 5. Cek tipe data dan data kosong
Ini penting supaya kita tahu apakah tanggal, angka, dan teks sudah benar.

In [ ]:
trx_detail.info()

In [ ]:
trx_detail.isna().sum().sort_values(ascending=False).head(10)

## 6. Analisis dasar: total penjualan
Untuk pemula, mulai dari pertanyaan sederhana:
- Berapa total penjualan?
- Berapa total qty terjual?
- Ada berapa transaksi unik?

In [ ]:
total_sales = trx_detail['Net Sales'].sum()
total_qty = trx_detail['Qty'].sum()
total_transaksi = trx_detail['No Transaksi'].nunique()

print(f'Total Net Sales : {total_sales:,.0f}')
print(f'Total Qty       : {total_qty:,.0f}')
print(f'Jumlah Transaksi: {total_transaksi:,.0f}')

## 7. Top 10 produk paling laku
Di sini kita kelompokkan berdasarkan `SKU`, lalu jumlahkan `Qty`.

In [ ]:
top_produk_qty = (
    trx_detail.groupby('SKU', as_index=False)['Qty']
    .sum()
    .sort_values('Qty', ascending=False)
    .head(10)
)

top_produk_qty

## 8. Menggabungkan dengan master produk (konsep seperti VLOOKUP)
Di Excel biasanya orang memakai **VLOOKUP/XLOOKUP**.
Di Python/Pandas, konsepnya setara dengan **merge**.

In [ ]:
top_produk_detail = top_produk_qty.merge(
    produk[['SKU', 'Nama Produk', 'Kategori', 'Brand', 'Harga Jual']],
    on='SKU',
    how='left'
)

top_produk_detail

## 9. Penjualan per toko
Sekarang kita lihat toko mana yang paling besar penjualannya.

In [ ]:
sales_per_toko = (
    trx_detail.groupby('Kode Toko', as_index=False)
    .agg(
        total_sales=('Net Sales', 'sum'),
        total_qty=('Qty', 'sum'),
        total_transaksi=('No Transaksi', 'nunique')
    )
    .sort_values('total_sales', ascending=False)
)

sales_per_toko = sales_per_toko.merge(
    toko[['Kode Toko', 'Nama Toko', 'Region', 'Tipe Toko']],
    on='Kode Toko',
    how='left'
)

sales_per_toko.head(10)

## 10. Visualisasi sederhana
Bar chart ini cocok untuk pemula karena mudah dibaca.

In [ ]:
plot_data = sales_per_toko.head(8).sort_values('total_sales')

plt.figure(figsize=(10,5))
plt.barh(plot_data['Nama Toko'], plot_data['total_sales'])
plt.title('Top Toko berdasarkan Net Sales')
plt.xlabel('Net Sales')
plt.ylabel('Nama Toko')
plt.tight_layout()
plt.show()

## 11. Analisis menengah: penjualan per kategori
Kita gabungkan detail transaksi dengan master produk, lalu buat ringkasan per kategori.

In [ ]:
trx_produk = trx_detail.merge(
    produk[['SKU', 'Nama Produk', 'Kategori', 'Brand', 'HPP', 'Harga Jual']],
    on='SKU',
    how='left'
)

kategori_summary = (
    trx_produk.groupby('Kategori', as_index=False)
    .agg(
        total_sales=('Net Sales', 'sum'),
        total_qty=('Qty', 'sum')
    )
    .sort_values('total_sales', ascending=False)
)

kategori_summary

## 12. Hitung estimasi gross profit
Karena ada `HPP`, kita bisa hitung pendekatan margin:

- `Estimasi COGS = Qty x HPP`
- `Estimasi Gross Profit = Net Sales - Estimasi COGS`

In [ ]:
trx_produk['Estimasi COGS'] = trx_produk['Qty'] * trx_produk['HPP']
trx_produk['Estimasi GP'] = trx_produk['Net Sales'] - trx_produk['Estimasi COGS']

gp_summary = (
    trx_produk.groupby('Kategori', as_index=False)
    .agg(
        net_sales=('Net Sales', 'sum'),
        cogs=('Estimasi COGS', 'sum'),
        gross_profit=('Estimasi GP', 'sum')
    )
)

gp_summary['gp_margin_pct'] = gp_summary['gross_profit'] / gp_summary['net_sales']
gp_summary.sort_values('gross_profit', ascending=False)

## 13. Analisis stok: produk yang perlu reorder
Sheet `Persediaan` biasanya dipakai tim operasional untuk memantau stok kritis.

In [ ]:
persediaan[['Kode Toko', 'Nama Toko', 'SKU', 'Nama Produk', 'Stok Akhir', 'Min Stock', 'Status', 'Saran Order']].head(10)

In [ ]:
reorder = persediaan[persediaan['Status'].astype(str).str.contains('reorder|critical|below', case=False, na=False)]

print('Jumlah item perlu perhatian stok:', len(reorder))
reorder.head(20)

## 14. Pivot table sederhana dengan Pandas
Ini versi Python dari pivot table Excel.

In [ ]:
pivot_kategori_toko = pd.pivot_table(
    trx_produk,
    index='Kategori',
    columns='Kode Toko',
    values='Net Sales',
    aggfunc='sum',
    fill_value=0
)

pivot_kategori_toko.head()

## 15. KPI dashboard mini
Bagian ini cocok untuk level menengah agar mulai berpikir seperti analyst.

In [ ]:
avg_basket = total_sales / total_transaksi if total_transaksi else 0
avg_item_per_trx = total_qty / total_transaksi if total_transaksi else 0
member_trx = trx_header['Member ID'].notna().sum() if 'Member ID' in trx_header.columns else 0
member_ratio = member_trx / len(trx_header) if len(trx_header) else 0

kpi = pd.DataFrame({
    'KPI': ['Total Net Sales', 'Total Qty', 'Total Transaksi', 'Average Basket Size', 'Average Item per Transaksi', 'Rasio Transaksi Member'],
    'Nilai': [total_sales, total_qty, total_transaksi, avg_basket, avg_item_per_trx, member_ratio]
})

kpi

## 16. Latihan mandiri
Agar cepat mahir, coba kerjakan latihan berikut:

### Level Awam
1. Cari 5 produk dengan `Net Sales` tertinggi.
2. Cari toko dengan jumlah transaksi terbanyak.
3. Hitung total penjualan per metode bayar.

### Level Menengah
1. Hitung gross profit per brand.
2. Buat ranking kategori per toko.
3. Bandingkan transaksi member vs non-member.
4. Buat analisis stok berdasarkan `Saran Order`.

## 17. Contoh jawaban latihan: penjualan per metode bayar

In [ ]:
sales_metode_bayar = (
    trx_detail.groupby('Metode Bayar', as_index=False)['Net Sales']
    .sum()
    .sort_values('Net Sales', ascending=False)
)
sales_metode_bayar

## 18. Kesimpulan
Setelah memahami notebook ini, Anda sudah belajar:
- membaca banyak sheet Excel
- memahami struktur data retail
- melakukan lookup/merge data
- membuat ringkasan penjualan
- menghitung KPI sederhana
- membuat pivot table dasar

Langkah berikutnya adalah belajar:
1. filtering yang lebih spesifik
2. analisis tanggal / bulanan
3. dashboard visual
4. forecasting sederhana